> **INSTRUCTOR SOLUTIONS** — do not share with learners before the session.

# Part 9 · Notebook 01 — Mean reversion, half-lives and Bayesian shrinkage

**Sessions:** S1 (Stochastic processes & Bayesian inference) · [Lesson plan](../../docs/lessons/PART_09_STATISTICAL_TRADING.md) · graded labs in [`labs/part09/`](../../labs/part09/)

**You will:**
1. Fit an Ornstein–Uhlenbeck process and read off its half-life.
2. See how uncertain a half-life is with one year vs five years of data.
3. Tell mean reversion from a random walk with the variance ratio.
4. Shrink a noisy mean return toward a prior.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
All data is synthetic with known parameters (true half-lives, true pairs, true hedge ratios, planted premia), so every estimator can be checked against the truth.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p9lib.py is in notebooks/part09/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p9lib as p

p.use_course_style()

## 1. The Ornstein–Uhlenbeck process

`dX = θ(μ − X)dt + σdW`: the further `X` is from `μ`, the harder it is pulled back. The **half-life** `ln 2 / θ` is how long a deviation takes to halve, the single most useful number in mean-reversion trading: it sets the look-back of the z-score and the time stop. Here are three paths with the same noise and different speeds.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.6))
for hl in (3, 15, 80):
    ax.plot(p.simulate_ou(500, np.log(2) / hl, 0.0, 1.0, seed=1), lw=1, label=f"half-life {hl} bars")
ax.axhline(0, color="black", lw=0.6); ax.legend(); ax.set_title("OU paths: faster reversion, tighter band"); plt.show()

## 2. Fitting it

The exact discretization is an AR(1): `X[t+1] = a + b·X[t] + e`. Regress with `np.polyfit(x[:-1], x[1:], 1)` (it returns `b, a`), then `θ = −ln(b)`, `μ = a/(1 − b)`, `σ = std(residuals, ddof=2)·√(2θ/(1 − b²))`, `half_life = ln 2/θ`. If `b` is not in `(0, 1)` the series doesn't mean-revert: return `θ = 0`, `half_life = inf` and NaN for `μ` and `σ`.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def fit_ou(x):
    x = np.asarray(x, dtype=float)
    b, a = np.polyfit(x[:-1], x[1:], 1)
    if not 0 < b < 1:
        return {"theta": 0.0, "mu": np.nan, "sigma": np.nan, "half_life": np.inf}
    resid = x[1:] - (a + b * x[:-1])
    theta = -np.log(b)
    return {"theta": theta, "mu": a / (1 - b), "sigma": resid.std(ddof=2) * np.sqrt(2 * theta / (1 - b ** 2)),
            "half_life": np.log(2) / theta}

x = p.simulate_ou(5000, np.log(2) / 7, 0.0, 1.0, seed=3)
rw = np.cumsum(np.random.default_rng(1).normal(size=2000))
mine = [p.attempt(fit_ou, x), p.attempt(fit_ou, rw)]
mine = p.check("fit_ou", mine, [p.fit_ou(x), p.fit_ou(rw)])
print(f"true half-life 7.0 → estimated {mine[0]['half_life']:.2f}; a random walk → {mine[1]['half_life']:.0f} bars")

The random walk's half-life comes out long but **finite**: in a finite sample the AR(1) slope is biased below 1, so every random walk looks a little mean-reverting. Always ask how uncertain the estimate is. A block bootstrap gives a 90% interval:

In [ ]:
for years in (1, 5):
    xx = p.simulate_ou(252 * years, np.log(2) / 7, 0.0, 1.0, seed=4)
    lo, hi = p.half_life_ci(xx)
    print(f"{years} year(s) of data: half-life {p.fit_ou(xx)['half_life']:.1f}, 90% interval {lo:.1f} to {hi:.1f}  (true 7.0)")

## 3. The variance ratio

For a random walk, the variance of `q`-step changes is `q` times that of 1-step changes. Mean reversion makes long-horizon changes smaller than that; trending makes them larger. `VR(q) = Var(x[t+q] − x[t]) / (q · Var(x[t+1] − x[t]))` (ddof=1): about 1 for a random walk, below 1 for mean reversion, above 1 for a trend.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def variance_ratio(x, q=5):
    x = np.asarray(x, dtype=float)
    return float(np.var(x[q:] - x[:-q], ddof=1) / (q * np.var(np.diff(x), ddof=1)))

trend = np.cumsum(p.simulate_ou(2000, np.log(2) / 5, 0.0, 1.0, seed=6))   # persistent increments: a trending series
cases = {"OU (half-life 7)": x, "random walk": rw, "trending": trend}
mine = {k: p.attempt(variance_ratio, v) for k, v in cases.items()}
mine = p.check("variance_ratio", mine, {k: p.variance_ratio(v) for k, v in cases.items()})
pd.Series(mine).round(3)

## 4. Bayesian shrinkage of a mean return

One year of daily returns says very little about the true mean. With a normal prior (mean `m0`, sd `s0`) and the data variance treated as known (`s² = var(r, ddof=1)`), the posterior is normal with

`precision = 1/s0² + n/s²` and `mean = (m0/s0² + n·x̄/s²) / precision`, sd `= √(1/precision)`.

The posterior mean sits between the prior and the sample mean, closer to whichever is more precise.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def normal_posterior_mean(returns, prior_mean=0.0, prior_sd=0.001):
    r = np.asarray(returns, dtype=float)
    n, s2 = r.size, r.var(ddof=1)
    prec = 1 / prior_sd ** 2 + n / s2
    return float((prior_mean / prior_sd ** 2 + n * r.mean() / s2) / prec), float(np.sqrt(1 / prec))

rng = np.random.default_rng(2)
one_year, ten_years = rng.normal(0.0004, 0.01, 252), rng.normal(0.0004, 0.01, 2520)
cases = [(one_year,), (ten_years,), (one_year, 0.0, 0.01)]
mine = [p.attempt(normal_posterior_mean, *c) for c in cases]
mine = p.check("normal_posterior_mean", mine, [p.normal_posterior_mean(*c) for c in cases])
pd.DataFrame({"sample mean (bp/day)": [c[0].mean() * 1e4 for c in cases], "posterior mean": [m * 1e4 for m, _ in mine],
              "posterior sd": [s * 1e4 for _, s in mine]}, index=["1 year, tight prior", "10 years, tight prior", "1 year, vague prior"]).round(2)

The true mean is 4 bp a day, but with 1% daily volatility even ten years pin it down only to about ±2 bp (this sample says 0.8). A skeptical prior ("most edges are near zero", sd 10 bp) shrinks one year's estimate by about a quarter (3.0 → 2.2 bp) and ten years' by only a few percent: the more data, the less the prior matters. With a vague prior (sd 100 bp), the posterior is simply the sample mean.

## Wrap-up

* The half-life is the unit of time for everything mean-reverting; estimate it with an interval.
* Variance ratios and half-lives are diagnostics, noisy on short samples.
* Shrink noisy estimates toward a sensible prior.
* Graded version: `labs/part09/week31_pairs` (OU, variance ratio, posterior, block-bootstrap half-life CI).